# Tamil Nadu Election Data Analysis

## Project Overview
End-to-end analysis of Tamil Nadu election candidate data using **Python and Pandas**. The analysis covers data quality, candidate performance, party performance, winners, winning margins, vote share, district-level patterns, and NOTA.

### Objectives
- Understand the structure and quality of the election data.
- Compare candidate and party-level voting performance.
- Identify winners and runner-ups by constituency.
- Calculate and categorize winning margins.
- Analyze vote share and NOTA separately from normal candidates.
- Produce concise, business-style insights for Power BI reporting.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Load the Dataset

The project uses the enriched candidate dataset and the party reference table. For GitHub, place the files in a `data/` folder.

In [ ]:
def find_file(filename):
    candidates = [
        Path('data') / filename,
        Path(filename),
        Path('../data') / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f'Could not find {filename}. Place it in the project data/ folder.'
    )

candidates_path = find_file('candidates_enriched.csv')
parties_path = find_file('parties.csv')

candidates = pd.read_csv(candidates_path)
parties = pd.read_csv(parties_path)

print(f'Candidates dataset: {candidates.shape[0]:,} rows × {candidates.shape[1]} columns')
print(f'Parties reference: {parties.shape[0]:,} rows × {parties.shape[1]} columns')

## 3. Data Overview

In [ ]:
display(candidates.head())
print('\nColumns:')
print(candidates.columns.tolist())

print('\nData types:')
display(candidates.dtypes.to_frame('dtype'))

## 4. Data Quality Checks

Check missing values, duplicate records, duplicate candidate IDs, and basic uniqueness.

In [ ]:
missing_summary = candidates.isna().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]

print('Missing values:')
display(missing_summary.to_frame('missing_count'))

print(f'Duplicate rows: {candidates.duplicated().sum():,}')
print(f'Duplicate candidate IDs: {candidates["candidate_id"].duplicated().sum():,}')
print(f'Unique constituency IDs: {candidates["constituency_id"].nunique():,}')
print(f'Unique parties: {candidates.loc[candidates["is_nota"] == 0, "party_id"].nunique():,}')

## 5. Separate Normal Candidates and NOTA

NOTA is not a political party or candidate, so it is analyzed separately from normal candidate performance.

In [ ]:
normal_candidates = candidates[candidates['is_nota'] == 0].copy()
nota = candidates[candidates['is_nota'] == 1].copy()

summary = pd.Series({
    'Total records': len(candidates),
    'Normal candidate records': len(normal_candidates),
    'NOTA records': len(nota),
    'Constituencies': candidates['constituency_id'].nunique(),
    'Parties (excluding NOTA)': normal_candidates['party_id'].nunique(),
    'Normal candidate votes': normal_candidates['total_votes'].sum(),
    'NOTA votes': nota['total_votes'].sum(),
    'Average votes per normal candidate': normal_candidates['total_votes'].mean(),
                
                })
display(summary.to_frame('value'))

## 6. Candidate Performance

Rank normal candidates by total votes.

In [ ]:
top_candidates = (
    normal_candidates[
        ['candidate_name', 'party', 'constituency_name', 'total_votes', 'vote_share_percent']
    ]
    .sort_values('total_votes', ascending=False)
    .head(10)
)

display(top_candidates)

## 7. Party Performance

Aggregate candidate-level results by party to compare overall voting performance.

In [ ]:
party_performance = (
    normal_candidates.groupby('party')
    .agg(
        candidate_count=('candidate_id', 'count'),
        total_votes=('total_votes', 'sum'),
        average_votes=('total_votes', 'mean'),
        highest_votes=('total_votes', 'max'),
        lowest_votes=('total_votes', 'min')
    )
)

party_performance['vote_range'] = (
    party_performance['highest_votes'] - party_performance['lowest_votes']
)
party_performance['vote_share_percent'] = (
    party_performance['total_votes'] / normal_candidates['total_votes'].sum() * 100
)
party_performance = party_performance.sort_values('total_votes', ascending=False)

display(party_performance.head(15))

### Party vote comparison

In [ ]:
top_party_plot = party_performance.head(10).sort_values('total_votes')

plt.figure(figsize=(10, 6))
plt.barh(top_party_plot.index, top_party_plot['total_votes'])
plt.xlabel('Total Votes')
plt.ylabel('Party')
plt.title('Top 10 Parties by Total Votes')
plt.tight_layout()
plt.show()

## 8. Winner Analysis

The winner in each constituency is identified from the source `status` field.

In [ ]:
winners = candidates[candidates['status'].str.lower() == 'won'].copy()

winner_summary = winners[
    ['constituency_id', 'constituency_name', 'candidate_name', 'party', 'total_votes', 'constituency_margin']
].sort_values('total_votes', ascending=False)

print(f'Winning records: {len(winners):,}')
display(winner_summary.head(10))

## 9. Winning Margin Analysis

Winning margin is the difference between the winner's votes and the runner-up's votes.

**Winning Margin = Winner Votes − Runner-up Votes**

In [ ]:
candidates_sorted = normal_candidates.sort_values(
    ['constituency_id', 'total_votes'],
    ascending=[True, False]
)

top_two = candidates_sorted.groupby('constituency_id').head(2).copy()

winner_runner = (
    top_two.groupby('constituency_id')
    .agg(
        constituency_name=('constituency_name', 'first'),
        winner=('candidate_name', 'first'),
        winner_party=('party', 'first'),
        winner_votes=('total_votes', 'first'),
        runner_up=('candidate_name', 'last'),
        runner_up_party=('party', 'last'),
        runner_up_votes=('total_votes', 'last')
    )
)

winner_runner['winning_margin'] = (
    winner_runner['winner_votes'] - winner_runner['runner_up_votes']
)

display(winner_runner.sort_values('winning_margin').head(10))

### Winning-margin categories

- **Close:** < 10,000 votes
- **Moderate:** 10,000–24,999 votes
- **Strong:** 25,000–49,999 votes
- **Landslide:** ≥ 50,000 votes

In [ ]:
def margin_category(margin):
    if margin >= 50_000:
        return 'Landslide'
    if margin >= 25_000:
        return 'Strong'
    if margin >= 10_000:
        return 'Moderate'
    return 'Close'

winner_runner['margin_category'] = winner_runner['winning_margin'].apply(margin_category)

margin_summary = (
    winner_runner['margin_category']
    .value_counts()
    .reindex(['Close', 'Moderate', 'Strong', 'Landslide'])
    .fillna(0)
    .astype(int)
)

display(margin_summary.to_frame('constituency_count'))

## 10. Vote Share Analysis

Compare winner vote shares across constituencies.

In [ ]:
winner_vote_share = winners[
    ['constituency_name', 'candidate_name', 'party', 'total_votes', 'vote_share_percent', 'constituency_margin']
].copy()

highest_vote_share = winner_vote_share.sort_values(
    'vote_share_percent', ascending=False
).head(10)

lowest_vote_share = winner_vote_share.sort_values(
    'vote_share_percent', ascending=True
).head(10)

print('Highest winner vote shares:')
display(highest_vote_share)

print('Lowest winner vote shares:')
display(lowest_vote_share)

## 11. District-Level Analysis

Analyze the distribution of constituency wins across districts.

In [ ]:
district_seats = (
    winners.groupby('district_name')['constituency_id']
    .nunique()
    .sort_values(ascending=False)
)

print('Districts by number of constituencies:')
display(district_seats.head(10).to_frame('constituencies'))

## 12. NOTA Analysis

In [ ]:
nota_analysis = (
    nota[['constituency_id', 'constituency_name', 'district_name', 'total_votes']]
    .sort_values('total_votes', ascending=False)
)

print(f'Total NOTA votes: {nota["total_votes"].sum():,}')
display(nota_analysis.head(15))

## 13. Key Findings

The following values are calculated directly from the source dataset used in this notebook.

In [ ]:
key_findings = {
    'Total records': len(candidates),
    'Normal candidate records': len(normal_candidates),
    'NOTA records': len(nota),
    'Total candidate votes': int(normal_candidates['total_votes'].sum()),
    'Average votes per normal candidate': round(normal_candidates['total_votes'].mean(), 2),
    'Parties excluding NOTA': normal_candidates['party_id'].nunique(),
    'Constituencies': candidates['constituency_id'].nunique(),
    'NOTA votes': int(nota['total_votes'].sum()),
    'Close-margin constituencies': int((winner_runner['winning_margin'] < 10_000).sum()),
}

display(pd.Series(key_findings, name='value').to_frame())

## 14. Conclusion

This analysis demonstrates an end-to-end Python/Pandas workflow for an election dataset: loading and validating data, separating NOTA records, analyzing candidate and party performance, identifying winners and runner-ups, calculating winning margins, analyzing vote share, and producing summary insights for business-style reporting and Power BI visualization.